# Week 4 — Memory Optimization in Distributed Training (ZeRO & FSDP)

**Course:** High-Performance Computing & Scaling Large Models  
**Practical:** DeepSpeed ZeRO-3 and dataset prefetching/caching.

---

## Learning Objectives

1. Decompose the per-GPU memory footprint of training into *model states* (parameters, gradients, optimizer states) and *residual states* (activations, buffers).
2. Explain the three ZeRO stages and the memory each one saves.
3. Compare ZeRO to PyTorch's native FSDP (FullyShardedDataParallel) and understand their algorithmic equivalence.
4. Configure DeepSpeed ZeRO-3 in code and reason about communication volume.
5. Use mixed-precision storage, gradient checkpointing, and offloading to push the achievable model size further.
6. Engineer a data pipeline that does not become the new bottleneck.


## 1. Accounting for Training Memory

For a model with $\Psi$ parameters trained with mixed-precision Adam, the per-GPU memory footprint of *model states* alone is:

| Component | Storage | Bytes per parameter |
|-----------|---------|---------------------|
| Parameters (FP16) | $2\Psi$ | 2 |
| Gradients (FP16) | $2\Psi$ | 2 |
| Optimizer master copy (FP32) | $4\Psi$ | 4 |
| Adam $m$ (FP32) | $4\Psi$ | 4 |
| Adam $v$ (FP32) | $4\Psi$ | 4 |
| **Total** | $\mathbf{16\Psi}$ | **16** |

A 7B-parameter model thus requires **112 GB** for model states — before any activations, before the KV cache, before the dataloader. No single commodity GPU has that capacity.

Plain **DistributedDataParallel (DDP)** replicates all 16 bytes per parameter on every GPU. Doubling the GPU count buys you twice the throughput but *zero* memory savings.

ZeRO removes the replication.


## 2. The ZeRO Family

Rajbhandari et al. (2020) observed that nothing forces every GPU to hold every component all the time. ZeRO **shards** model states across the data-parallel group, **gathering** them on demand and **scattering** updates back.

| Stage | Shards | Per-GPU memory (with $N$ GPUs) | Communication |
|-------|--------|--------------------------------|---------------|
| **ZeRO-1** | Optimizer states | $4\Psi + \tfrac{12\Psi}{N}$ | Same as DDP |
| **ZeRO-2** | + Gradients | $2\Psi + \tfrac{14\Psi}{N}$ | Same as DDP |
| **ZeRO-3** | + Parameters | $\tfrac{16\Psi}{N}$ | ~1.5× DDP |

**Why ZeRO-3 costs more bandwidth.** Parameters live partitioned. To run the forward pass of layer $\ell$, every GPU must `all-gather` that layer's parameters, use them, then drop them. The backward pass does the same. The extra `all-gather`s account for the bandwidth overhead — but on NVLink-connected nodes, it is usually negligible.

### ZeRO-Offload and ZeRO-Infinity

For workloads where even ZeRO-3 cannot fit, optimizer states (and optionally parameters) can be **offloaded to CPU memory or NVMe SSDs**. The arithmetic ratio in Adam is favorable: the optimizer update is a small fraction of total work, so paying its latency over PCIe is often acceptable.

These tiers are stacked: ZeRO-3 + activation checkpointing + CPU offload + NVMe offload routinely enables training of 100B+ parameter models on a single 8-GPU node.


## 3. FSDP: ZeRO-3 in PyTorch

PyTorch's `torch.distributed.fsdp.FullyShardedDataParallel` (FSDP) implements the same algorithm as ZeRO-3, integrated natively. Conceptually:

```
Forward:
    for each FSDP-wrapped unit:
        all_gather(parameters)           # rebuild full weights
        out = unit(input)                # ordinary forward
        free(parameters)                 # back to sharded state

Backward:
    for each FSDP-wrapped unit:
        all_gather(parameters)
        unit.backward(grad)
        reduce_scatter(gradients)        # average and shard
        free(parameters); free(grads)
```

The unit of sharding is an `FSDP` module. Wrapping the whole model in one `FSDP` is correct but bad: parameters live in *one giant shard*, and the all-gather of every forward step transfers the entire model. Best practice is to wrap *transformer blocks* individually, so only one block's parameters are gathered at a time.

Below is a working FSDP setup. We use a small model so the cell runs on any 2-GPU node. For single-GPU machines, the code paths still execute (`world_size=1`), and you can inspect the wrapping behavior.


In [ ]:
import os, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}, devices = {torch.cuda.device_count()}")


In [ ]:
# A small transformer for distributed-training experiments
class Block(nn.Module):
    def __init__(self, d=512, h=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ln1  = nn.LayerNorm(d)
        self.ln2  = nn.LayerNorm(d)
        self.mlp  = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        x    = self.ln1(x + a)
        x    = self.ln2(x + self.mlp(x))
        return x


class SmallLM(nn.Module):
    def __init__(self, vocab=10000, d=512, n_layers=6, n_heads=8, max_len=512):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(max_len, d)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.ln    = nn.LayerNorm(d)
        self.head  = nn.Linear(d, vocab, bias=False)

    def forward(self, ids):
        T   = ids.size(1)
        pos = torch.arange(T, device=ids.device)
        x   = self.tok(ids) + self.pos(pos)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.ln(x))


def model_params(m):
    return sum(p.numel() for p in m.parameters())

m = SmallLM()
print(f"Model parameters: {model_params(m)/1e6:.2f} M")


### 3.1 FSDP Wrapping

The cell below shows the *recommended* FSDP wrapping policy. We use an `auto_wrap_policy` that wraps each `Block` separately, so that activations from earlier layers don't pin all of the model's parameters in memory.


In [ ]:
# FSDP setup. We do NOT spawn a process group here so the notebook works
# stand-alone. To run FSDP for real, launch the script in `scripts/train_fsdp.py`
# with `torchrun --nproc_per_node=2 scripts/train_fsdp.py`.
import functools
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import MixedPrecision, ShardingStrategy
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
import torch.distributed as dist

def fsdp_wrap_policy():
    return functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={Block},
    )


def build_fsdp_model(rank, world_size, d=512):
    """Construct an FSDP-wrapped model on a given rank."""
    torch.cuda.set_device(rank)
    model = SmallLM(d=d).cuda(rank)

    mp_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )

    fsdp_model = FSDP(
        model,
        auto_wrap_policy=fsdp_wrap_policy(),
        sharding_strategy=ShardingStrategy.FULL_SHARD,  # ZeRO-3 equivalent
        mixed_precision=mp_policy,
        device_id=torch.cuda.current_device(),
    )
    return fsdp_model


# Print the wrapping decisions without launching distributed.
print("FSDP wrapping policy: each Block becomes a separate FSDP unit.")
print("Each block gathers its parameters in forward/backward, then releases them.")


### 3.2 Estimating Memory Savings

We can predict per-GPU memory analytically *before* launching a job. The function below estimates model-state memory for DDP, ZeRO-1, ZeRO-2, and ZeRO-3 as a function of the world size.


In [ ]:
def memory_breakdown(Psi, world_size):
    """Memory in bytes for model states under Adam mixed precision."""
    # Bytes per parameter: 2 (fp16 weights) + 2 (fp16 grads) + 4 (fp32 master) + 4 + 4 (m, v)
    base = 16
    fp16_weights = 2 * Psi
    fp16_grads   = 2 * Psi
    fp32_states  = 12 * Psi   # master copy + m + v
    return {
        "DDP"    : (fp16_weights + fp16_grads + fp32_states),
        "ZeRO-1" : (fp16_weights + fp16_grads + fp32_states / world_size),
        "ZeRO-2" : (fp16_weights + fp16_grads / world_size + fp32_states / world_size),
        "ZeRO-3" : (fp16_weights / world_size + fp16_grads / world_size + fp32_states / world_size),
    }


print(f"Per-GPU model-state memory for a 7B model (Adam, mixed precision):\n")
print(f"{'N GPUs':>7} | {'DDP':>10} | {'ZeRO-1':>10} | {'ZeRO-2':>10} | {'ZeRO-3':>10}")
print("-" * 60)
for N in (1, 2, 4, 8, 16, 64):
    mem = memory_breakdown(7e9, N)
    fmt = lambda b: f"{b / 1e9:>7.1f} GB"
    print(f"{N:>7} | {fmt(mem['DDP'])} | {fmt(mem['ZeRO-1'])} | "
          f"{fmt(mem['ZeRO-2'])} | {fmt(mem['ZeRO-3'])}")


**Observations.**

- DDP is constant in GPU count: replication means each GPU still needs all 16 bytes/param.
- ZeRO-1 immediately wipes out the optimizer-state cost (12 of the 16 bytes).
- ZeRO-3 scales linearly to zero — at 64 GPUs, a 7B model costs about 1.75 GB of model state per GPU.

In practice, residual activations come back into the picture and you cannot scale forever. But this table explains why ZeRO is the *default* for any training above a few billion parameters.


## 4. DeepSpeed ZeRO-3 Configuration

DeepSpeed offers ZeRO via a JSON config + a thin Python wrapper around the PyTorch model. The configuration below corresponds to ZeRO-3 with parameter and optimizer offloading to CPU.


In [ ]:
# A minimal DeepSpeed ZeRO-3 configuration. Save as configs/zero3.json.
ds_config = {
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps": 4,
    "gradient_clipping": 1.0,
    "fp16": {"enabled": False},
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param":     {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 5e8,
        "stage3_prefetch_bucket_size": 5e8,
        "stage3_param_persistence_threshold": 1e6,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True,
    },
    "wall_clock_breakdown": False,
}

import json
print(json.dumps(ds_config, indent=2))


**Key knobs.**

- `stage3_prefetch_bucket_size` — DeepSpeed `all-gather`s the *next* shard while computing on the current one. Larger values overlap more communication but cost peak memory.
- `stage3_param_persistence_threshold` — small parameters (LayerNorm scales, biases) are kept resident rather than re-gathered every step.
- `offload_param: cpu` — the most aggressive setting; usable when training is bottlenecked by GPU memory but you have abundant host RAM and PCIe headroom.

A complete launch command for a single 8-GPU node:

```bash
deepspeed --num_gpus=8 scripts/train_deepspeed.py \
          --deepspeed configs/zero3.json \
          --model_size 7B \
          --dataset /data/c4
```


## 5. Activation Memory and Gradient Checkpointing

Even with weights sharded, *activations* can still dominate. For a transformer with $L$ layers, batch $B$, sequence length $S$, hidden dim $d$, activations scale roughly as $O(L \cdot B \cdot S \cdot d)$. A 24-layer model with $B=8, S=4096, d=4096$ stores roughly **6 GB** of activations.

**Gradient (activation) checkpointing** trades compute for memory: during forward we save only a *subset* of activations (typically one per transformer block); during backward we *recompute* the rest. Cost: ~33 % extra wall-clock; benefit: $O(\sqrt L)$ activation memory instead of $O(L)$.


In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedBlock(nn.Module):
    """Wraps a Block so its forward pass is checkpointed."""
    def __init__(self, block: Block):
        super().__init__()
        self.block = block

    def forward(self, x):
        return checkpoint(self.block, x, use_reentrant=False)


# Compare activation memory: with vs without checkpointing.
def measure_peak(model, x):
    torch.cuda.reset_peak_memory_stats()
    out  = model(x)
    loss = out.sum()
    loss.backward()
    return torch.cuda.max_memory_allocated() / 1e6


if torch.cuda.is_available():
    device = torch.device("cuda")
    B, T, D = 4, 256, 512
    x  = torch.randint(0, 10000, (B, T), device=device)

    base   = SmallLM(d=D, n_layers=8).to(device)
    mem_a  = measure_peak(base, x)

    ckpt = SmallLM(d=D, n_layers=8).to(device)
    ckpt.blocks = nn.ModuleList([CheckpointedBlock(b) for b in ckpt.blocks])
    mem_b  = measure_peak(ckpt, x)

    print(f"Baseline peak memory       : {mem_a:8.1f} MB")
    print(f"With activation checkpoint : {mem_b:8.1f} MB")
    print(f"Reduction                  : {(1 - mem_b / mem_a) * 100:5.1f} %")
else:
    print("CUDA required; skipping memory measurement.")


## 6. The Other Half of the Story: The Data Pipeline

A perfectly optimized model means nothing if your dataloader stalls. The diagnostic is simple: if `nvidia-smi dmon` shows GPU utilization regularly dipping below 80 %, you have a data problem.

### Common Failure Modes

- **CPU decode bottleneck.** JPEG decoding on the CPU at 5000 img/s is fine for one GPU and a death sentence for eight. Solutions: pre-decode to a binary format (Webdataset, MosaicML's MDS); use NVIDIA DALI to decode on the GPU.
- **Disk I/O bottleneck.** Random reads on rotational disks devastate throughput. Pack data into large *shards* (1–4 GB each) so reads become sequential.
- **Tokenization on the fly.** Tokenizing a 100 GB text corpus on every epoch wastes hours. Tokenize once, cache to disk as a binary `.bin` file with `numpy.memmap`.
- **Insufficient `num_workers`.** Each worker is a Python process; on a 32-core machine, set `num_workers=16` or higher and `prefetch_factor=4`.

### Tokenization Caching: A Quick Pattern


In [ ]:
# Pattern: tokenize once, cache to a memory-mapped binary file.
import numpy as np
from pathlib import Path

def tokenize_and_cache(text_iter, tokenizer, cache_path: str,
                       dtype=np.uint16):
    """
    Stream `text_iter` (an iterable of strings), tokenize, append to a single
    `.bin` file. Returns a numpy memmap view ready for training.
    """
    path = Path(cache_path)
    if path.exists():
        return np.memmap(path, dtype=dtype, mode="r")

    # First pass — count tokens and stream-write. For brevity, we use a
    # two-pass approach; a production version would write incrementally.
    all_ids = []
    for text in text_iter:
        all_ids.extend(tokenizer(text))
    arr = np.array(all_ids, dtype=dtype)
    arr.tofile(path)
    print(f"Wrote {len(arr):,} tokens ({path.stat().st_size / 1e6:.1f} MB) → {path}")
    return np.memmap(path, dtype=dtype, mode="r")


# Toy demonstration with a fake tokenizer.
def fake_tokenizer(s: str):
    return [hash(w) % 50000 for w in s.split()]


texts = [
    "High performance computing is the engine behind modern science.",
    "FlashAttention restructures attention to eliminate quadratic memory.",
    "ZeRO shards optimizer states across the data parallel group.",
] * 100

cache_path = "/tmp/toy_corpus.bin"
Path(cache_path).unlink(missing_ok=True)
arr = tokenize_and_cache(texts, fake_tokenizer, cache_path)
print(f"Memmap shape: {arr.shape}, dtype: {arr.dtype}")
print(f"First 16 tokens: {arr[:16].tolist()}")


**Why this works.**

- `numpy.memmap` lets the OS handle paging; you can iterate over a 1 TB corpus with constant RAM.
- Once written, the cache is *cheap*: every subsequent epoch is sequential disk I/O at line rate (~3 GB/s on NVMe), parsed by the kernel.
- `dtype=np.uint16` halves disk footprint vs `int32`. For vocabularies under 65 536, no information is lost.

Combined with a `DataLoader` using `num_workers=16, persistent_workers=True, prefetch_factor=4`, this pattern eliminates the data pipeline as a bottleneck for almost every research-scale workload.


## 7. Exercises

**Exercise 4.1 — Memory accounting.**  
Repeat the calculation of Section 1 for **Lion**, **Adafactor**, and **SGD with momentum**. Which optimizer benefits most from ZeRO-1 vs ZeRO-2?

**Exercise 4.2 — FSDP wrapping policy.**  
Modify the wrap policy to wrap *pairs of blocks* (`{Block, Block}`) as a single unit. Measure the effect on peak memory and on step time. Why might either go up?

**Exercise 4.3 — DeepSpeed inference of memory.**  
Use `deepspeed.estimate_zero3_model_states_mem_needs` to estimate the memory required to train a 13B model on 16× A100-80GB. Compare to your analytical estimate.

**Exercise 4.4 — Activation checkpointing granularity.**  
Implement checkpointing at three granularities: (a) every block, (b) every 2 blocks, (c) only the MLP within each block. Measure peak memory and step time for each.

**Exercise 4.5 — Dataloader profiling.**  
Add `prof.step()` calls inside a training loop and inspect the `aten::copy_` time. If the dataloader is starving the GPU, this kernel will appear at the top.


## 8. Summary

The per-GPU memory cost of training is dominated by *model states* (parameters, gradients, optimizer state) and *activations*. ZeRO shards the model states across the data-parallel group, and FSDP is its PyTorch-native equivalent. Gradient checkpointing trades a small wall-clock cost for square-root activation memory. Together, these techniques routinely let an 8-GPU node train models that, on any individual device, would not fit.

But sharding model state is not enough at the trillion-parameter scale. Week 5 introduces *tensor* and *pipeline* parallelism, which split the model *along its operations* rather than along its parameters — the recipe behind GPT-3, PaLM, and modern open-source giants.
